# LeetCode 726 — Number of Atoms bằng Python

- Link: https://leetcode.com/problems/number-of-atoms/
- Độ khó: **Hard**
- Chủ đề: Chuỗi (string), Ngăn xếp (stack), Đệ quy / Recursive Descent Parsing, Hash Map (`Counter`), Sắp xếp

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def countOfAtoms(self, formula: str) -> str:
```

## 1. Giới thiệu & mục tiêu học

Bài này yêu cầu ta **đọc một công thức hoá học** như `K4(ON(SO3)2)2` và đếm mỗi nguyên tố xuất hiện bao nhiêu lần. Nghe giống bài hoá, nhưng thực chất đây là bài **phân tích cú pháp (parsing)** — kỹ năng nền để viết máy tính bỏ túi, trình đọc JSON, trình biên dịch nhỏ…

Khó ở chỗ: **dấu ngoặc lồng nhau** và **hệ số nhân đứng sau ngoặc**. Khi đọc từ trái sang phải, ta gặp nội dung trong ngoặc *trước* khi biết nó bị nhân với bao nhiêu.

Sau notebook này bạn sẽ:

1. Biết cách "quét" một chuỗi bằng con trỏ `i`: đọc tên nguyên tố (1 chữ hoa + các chữ thường), đọc số nhiều chữ số.
2. Viết lời giải baseline "khai triển chuỗi" để làm trọng tài kiểm tra.
3. Viết **đệ quy (recursive descent)**: mỗi lần gặp `(` gọi đệ quy, gặp `)` trả về bộ đếm.
4. Viết lời giải **ngăn xếp các `Counter`** và lời giải **quét ngược với ngăn xếp hệ số**.
5. Kiểm chứng mọi cách bằng test ngẫu nhiên có seed, và nắm các lỗi người mới hay gặp (số nhiều chữ số, hệ số mặc định 1, sắp xếp đầu ra).

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho chuỗi `formula` là một công thức hoá học hợp lệ. Trả về chuỗi mô tả số lượng từng nguyên tử.

Quy tắc ngữ pháp:

- **Tên nguyên tố**: một chữ cái **in hoa**, theo sau là 0 hoặc nhiều chữ **thường**. Ví dụ: `H`, `O`, `Mg`, `Uuo`.
- Sau tên nguyên tố có thể có **số đếm** (≥ 2). Nếu số đếm là 1 thì **không viết**. Ví dụ `H2O` hợp lệ, `H1O2` không hợp lệ.
- Hai công thức viết liền nhau tạo thành công thức mới: `H2O2He3Mg4`.
- Công thức đặt trong ngoặc `( ... )` rồi có thể có số đếm phía sau: `(H2O2)`, `(H2O2)3`.

**Đầu ra**: các nguyên tố theo thứ tự **từ điển (sorted)**, mỗi nguyên tố viết tên, sau đó là số lượng nếu số lượng > 1.

**Ví dụ**

```text
Ví dụ 1: formula = "H2O"            -> "H2O"
Ví dụ 2: formula = "Mg(OH)2"        -> "H2MgO2"
Ví dụ 3: formula = "K4(ON(SO3)2)2"  -> "K4N2O14S4"
```

Giải thích ví dụ 3: `(SO3)2` = S2 O6. `ON(SO3)2` = O1 N1 S2 O6 = N1 O7 S2. Nhân 2: N2 O14 S4. Cộng K4 → `K4N2O14S4`.

**Ràng buộc**

- `1 <= formula.length <= 1000`
- `formula` gồm chữ cái tiếng Anh, chữ số, `(` và `)`.
- `formula` luôn hợp lệ.
- Kết quả số lượng vừa kiểu int 32-bit.

In [1]:
EXAMPLES = [
    ("H2O", "H2O"),
    ("Mg(OH)2", "H2MgO2"),
    ("K4(ON(SO3)2)2", "K4N2O14S4"),
]
for f, expected in EXAMPLES:
    print(f"{f:<16} -> mong đợi {expected}")

H2O              -> mong đợi H2O
Mg(OH)2          -> mong đợi H2MgO2
K4(ON(SO3)2)2    -> mong đợi K4N2O14S4


## 3. Kiến thức nền

### 3.1. Phân loại ký tự: `isupper`, `islower`, `isdigit`

Mọi bộ phân tích cú pháp đều bắt đầu bằng việc nhìn **một ký tự** và quyết định nó thuộc loại gì.

In [2]:
for ch in "Mg(OH)12":
    print(repr(ch), "hoa" if ch.isupper() else "", "thường" if ch.islower() else "",
          "số" if ch.isdigit() else "", "ngoặc" if ch in "()" else "")

'M' hoa   
'g'  thường  
'('    ngoặc
'O' hoa   
'H' hoa   
')'    ngoặc
'1'   số 
'2'   số 


### 3.2. Đọc một "token" bằng con trỏ `i`

Kỹ thuật cốt lõi: giữ biến `i` chỉ vào vị trí đang đọc. Muốn đọc một số nhiều chữ số, ta "nuốt" các chữ số liên tiếp:

```python
j = i
while j < n and s[j].isdigit():
    j += 1
num = int(s[i:j]) if j > i else 1   # không có số -> mặc định 1
i = j
```

Đọc tên nguyên tố tương tự: 1 chữ hoa, rồi nuốt các chữ thường.

In [3]:
def read_name(s, i):
    j = i + 1                      # s[i] chắc chắn là chữ hoa
    while j < len(s) and s[j].islower():
        j += 1
    return s[i:j], j

def read_num(s, i):
    j = i
    while j < len(s) and s[j].isdigit():
        j += 1
    return (int(s[i:j]) if j > i else 1), j

s = "Uuo12Mg"
name, i = read_name(s, 0); print(name, i)
num, i = read_num(s, i);   print(num, i)
name, i = read_name(s, i); print(name, i)
num, i = read_num(s, i);   print(num, i, "(không có số -> 1)")

Uuo 3
12 5
Mg 7
1 7 (không có số -> 1)


### 3.3. `collections.Counter` — bộ đếm

`Counter` là một `dict` đặc biệt: truy cập khoá chưa có trả về `0` (không lỗi), và cộng hai `Counter` được bằng `+` hoặc `update`.

In [4]:
from collections import Counter

c = Counter()
c["H"] += 2
c["O"] += 1
print(c, c["N"])            # khoá chưa có -> 0

d = Counter({"H": 1, "C": 3})
c.update(d)                 # cộng dồn
print(c)

# nhân mọi giá trị với k
k = 3
print({atom: cnt * k for atom, cnt in c.items()})

Counter({'H': 2, 'O': 1}) 0
Counter({'H': 3, 'C': 3, 'O': 1})
{'H': 9, 'O': 3, 'C': 9}


### 3.4. Ngăn xếp (stack) bằng `list`

`append` đẩy vào đỉnh, `pop` lấy ra từ đỉnh, `stack[-1]` xem đỉnh. Ngăn xếp rất hợp với **ngoặc lồng nhau**: mở ngoặc → đẩy, đóng ngoặc → lấy ra.

In [5]:
stack = []
for ch in "(()(()))":
    if ch == "(":
        stack.append(len(stack))
        print("mở  -> độ sâu", len(stack))
    else:
        stack.pop()
        print("đóng -> độ sâu", len(stack))

mở  -> độ sâu 1
mở  -> độ sâu 2
đóng -> độ sâu 1
mở  -> độ sâu 2
mở  -> độ sâu 3
đóng -> độ sâu 2
đóng -> độ sâu 1
đóng -> độ sâu 0


### 3.5. Định dạng đầu ra: sắp xếp và ghép chuỗi

`sorted(dict)` trả về các khoá theo thứ tự từ điển. Chú ý: so sánh chuỗi theo mã ký tự, nên `"H" < "He" < "Mg" < "O"`. Ghép bằng `"".join(...)` nhanh hơn cộng chuỗi trong vòng lặp.

In [6]:
cnt = {"O": 2, "H": 2, "Mg": 1, "He": 1}
parts = []
for atom in sorted(cnt):
    parts.append(atom + (str(cnt[atom]) if cnt[atom] > 1 else ""))
print(sorted(cnt), "".join(parts))

['H', 'He', 'Mg', 'O'] H2HeMgO2


### 3.6. Biểu thức chính quy (regex) để tách token

`re.findall` với các nhóm `(...)` trả về tuple cho từng lần khớp. Mẫu dưới đây tách một công thức thành các token: (tên nguyên tố, số) hoặc `(` hoặc `)` kèm số.

In [7]:
import re
TOKEN = re.compile(r"([A-Z][a-z]*)(\d*)|(\()|(\))(\d*)")
print(TOKEN.findall("Mg(OH)12"))

[('Mg', '', '', '', ''), ('', '', '(', '', ''), ('O', '', '', '', ''), ('H', '', '', '', ''), ('', '', '', ')', '12')]


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Ngữ pháp của công thức

Viết dạng ngữ pháp (giống cách mô tả ngôn ngữ lập trình):

```text
formula := part*
part    := ATOM COUNT?  |  "(" formula ")" COUNT?
ATOM    := [A-Z][a-z]*
COUNT   := [0-9]+        (không có thì = 1)
```

Ngữ pháp **tự tham chiếu** (`formula` chứa `formula` bên trong ngoặc) → gợi ý mạnh mẽ dùng **đệ quy** hoặc **ngăn xếp**.

### 4.2. Tính tay `Mg(OH)2`

| Bước | Đọc | Hành động | Bộ đếm hiện tại |
|---|---|---|---|
| 1 | `Mg` (không số) | Mg += 1 | {Mg:1} |
| 2 | `(` | mở bộ đếm mới cho trong ngoặc | ngoài {Mg:1}, trong {} |
| 3 | `O` | O += 1 (trong) | trong {O:1} |
| 4 | `H` | H += 1 (trong) | trong {O:1, H:1} |
| 5 | `)2` | nhân trong với 2, cộng vào ngoài | {Mg:1, O:2, H:2} |

Kết quả sắp xếp: `H2MgO2`.

### 4.3. Quan sát quan trọng

Mỗi nguyên tử được nhân với **tích các hệ số của mọi cặp ngoặc bao quanh nó**, rồi nhân với số đứng ngay sau nó. Ví dụ trong `K4(ON(SO3)2)2`, nguyên tử `O` trong `SO3` có hệ số `3 × 2 × 2 = 12`; `O` đứng sau `(` đầu tiên có hệ số `1 × 2 = 2`. Tổng O = 12 + 2 = 14. ✔

In [8]:
# kiểm tra quan sát 4.3 bằng tay
O_inner = 3 * 2 * 2
O_outer = 1 * 2
print("O =", O_inner + O_outer)

O = 14


## 5. Cách 1 — Brute Force: khai triển chuỗi

**Ý tưởng.** Cách "ngây thơ" nhất: biến công thức thành **danh sách phẳng các nguyên tử** bằng cách lặp lại thật sự. `(OH)2` → `["O","H","O","H"]`. Sau đó đếm bằng `Counter`.

Ta làm đệ quy: hàm `expand(i)` đọc từ vị trí `i` đến khi gặp `)` hoặc hết chuỗi, trả về danh sách nguyên tử và vị trí kết thúc.

In [9]:
def expand(formula):
    n = len(formula)

    def go(i):
        atoms = []
        while i < n and formula[i] != ")":
            if formula[i] == "(":
                inner, i = go(i + 1)     # i bây giờ chỉ vào ')'
                i += 1                   # bỏ qua ')'
                k, i = read_num(formula, i)
                atoms.extend(inner * k)  # lặp danh sách k lần
            else:
                name, i = read_name(formula, i)
                k, i = read_num(formula, i)
                atoms.extend([name] * k)
        return atoms, i

    return go(0)[0]

def format_counts(cnt):
    return "".join(a + (str(cnt[a]) if cnt[a] > 1 else "") for a in sorted(cnt))

def count_atoms_brute(formula):
    return format_counts(Counter(expand(formula)))

print(expand("Mg(OH)2"))
for f, e in EXAMPLES:
    print(f, count_atoms_brute(f), count_atoms_brute(f) == e)

['Mg', 'O', 'H', 'O', 'H']
H2O H2O True
Mg(OH)2 H2MgO2 True
K4(ON(SO3)2)2 K4N2O14S4 True


**Độ phức tạp.** Độ dài danh sách khai triển bằng **tổng số nguyên tử**, có thể rất lớn: `((((H99)99)99)99)` có `99^4 ≈ 9.6×10^7` nguyên tử! Thời gian và bộ nhớ tỉ lệ với con số này → **quá chậm** cho ràng buộc đề (kết quả có thể tới `2^31`).

**Nút thắt.** Ta lưu từng nguyên tử riêng lẻ, trong khi chỉ cần lưu **(tên, số lượng)**. Thay vì lặp danh sách `k` lần, hãy **nhân số lượng với `k`**.

In [10]:
import time
f = "((H9)9)9"
t = time.perf_counter(); r = count_atoms_brute(f); t = time.perf_counter() - t
print(r, f"mất {t*1000:.2f} ms, danh sách dài {len(expand(f))}")

H729 mất 0.06 ms, danh sách dài 729


## 6. Bước đột phá

**Insight.** Mọi nhóm (trong ngoặc hay toàn công thức) có thể được tóm tắt bằng một **bộ đếm** `{nguyên tố: số lượng}`. Hai phép toán cần thiết:

1. **Cộng** hai bộ đếm (hai phần viết liền nhau).
2. **Nhân** một bộ đếm với số `k` (phần trong ngoặc có hệ số `k`).

Kích thước bộ đếm ≤ số nguyên tố khác nhau (≤ độ dài chuỗi), **không phụ thuộc** vào số nguyên tử thật.

**Quy tắc nếu… thì…** khi đọc từ trái sang phải:

- Nếu gặp **chữ hoa** → đọc tên + số, cộng vào bộ đếm **hiện tại** (đỉnh ngăn xếp).
- Nếu gặp `(` → **mở** bộ đếm mới (đẩy vào ngăn xếp / gọi đệ quy).
- Nếu gặp `)` → đọc số `k` ngay sau, **lấy** bộ đếm đỉnh ra, nhân `k`, cộng vào bộ đếm bên dưới.

**Bất biến (invariant).** Tại mọi thời điểm, ngăn xếp có đúng `1 + (số ngoặc đang mở)` bộ đếm; bộ đếm thứ `d` chứa tổng nguyên tử của **phần đã đọc** ở độ sâu `d` (chưa nhân hệ số ngoài).

**Ý tưởng khác (quét ngược).** Nếu đọc từ **phải sang trái**, ta gặp hệ số **trước** nội dung ngoặc! Khi đó chỉ cần giữ một ngăn xếp các **hệ số tích luỹ**, và mỗi nguyên tử được cộng trực tiếp `count × tích_hiện_tại` vào một bộ đếm duy nhất.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đệ quy (Recursive Descent)

**Giải thích.** Viết hàm `parse(i)` đọc một `formula` bắt đầu từ `i`, dừng khi gặp `)` hoặc hết chuỗi; trả về `(Counter, i_mới)`. Mỗi khi gặp `(`, gọi `parse(i+1)` để lấy bộ đếm bên trong, sau đó đọc hệ số và **nhân**.

Đây chính là lời giải brute force, chỉ thay "danh sách" bằng "Counter" và "lặp k lần" bằng "nhân k". Nhờ có brute force, ta thấy rõ bước tối ưu là gì.

In [11]:
def count_atoms_recursive(formula):
    n = len(formula)

    def parse(i):
        cnt = Counter()
        while i < n and formula[i] != ")":
            if formula[i] == "(":
                inner, i = parse(i + 1)
                i += 1                              # bỏ qua ')'
                k, i = read_num(formula, i)
                for atom, c in inner.items():
                    cnt[atom] += c * k
            else:
                name, i = read_name(formula, i)
                k, i = read_num(formula, i)
                cnt[name] += k
        return cnt, i

    return format_counts(parse(0)[0])

for f, e in EXAMPLES:
    print(f, count_atoms_recursive(f), count_atoms_recursive(f) == e)
print(count_atoms_recursive("((((H99)99)99)99)"))

H2O H2O True
Mg(OH)2 H2MgO2 True
K4(ON(SO3)2)2 K4N2O14S4 True
H96059601


**Code nộp LeetCode (Cách 2).** Độ sâu đệ quy ≤ số ngoặc lồng nhau ≤ 500 (chuỗi 1000 ký tự) — dưới giới hạn mặc định 1000 của Python, nhưng tăng `setrecursionlimit` cho chắc.

```python
import sys
from collections import Counter

class Solution:
    def countOfAtoms(self, formula: str) -> str:
        sys.setrecursionlimit(10000)
        n = len(formula)
        self.i = 0

        def read_num():
            j = self.i
            while self.i < n and formula[self.i].isdigit():
                self.i += 1
            return int(formula[j:self.i]) if self.i > j else 1

        def parse():
            cnt = Counter()
            while self.i < n and formula[self.i] != ')':
                if formula[self.i] == '(':
                    self.i += 1
                    inner = parse()
                    self.i += 1               # bỏ ')'
                    k = read_num()
                    for a, c in inner.items():
                        cnt[a] += c * k
                else:
                    j = self.i
                    self.i += 1
                    while self.i < n and formula[self.i].islower():
                        self.i += 1
                    name = formula[j:self.i]
                    cnt[name] += read_num()
            return cnt

        cnt = parse()
        return ''.join(a + (str(cnt[a]) if cnt[a] > 1 else '') for a in sorted(cnt))
```

**Độ phức tạp.** Mỗi ký tự được đọc một lần, nhưng mỗi lần đóng ngoặc ta duyệt bộ đếm bên trong (kích thước ≤ `k` nguyên tố khác nhau). Trường hợp xấu `O(n · k)` với `n` độ dài, `k` số nguyên tố khác nhau; cộng thêm `O(k log k)` sắp xếp. Bộ nhớ `O(n · k)` trong trường hợp xấu (mỗi tầng một bộ đếm), thực tế nhỏ.

### 7.2. Cách 3 — Ngăn xếp các `Counter` (không đệ quy)

**Giải thích.** Thay vì để Python giữ "ngăn xếp lời gọi", ta tự giữ `stack` gồm các `Counter`. Ban đầu `stack = [Counter()]`.

- Chữ hoa → `stack[-1][name] += k`.
- `(` → `stack.append(Counter())`.
- `)` + số `k` → `top = stack.pop()`; với mỗi `(a, c)` trong `top`: `stack[-1][a] += c * k`.

Cuối cùng `stack[0]` là đáp án.

In [12]:
def count_atoms_stack(formula):
    n, i = len(formula), 0
    stack = [Counter()]
    while i < n:
        ch = formula[i]
        if ch == "(":
            stack.append(Counter())
            i += 1
        elif ch == ")":
            k, i = read_num(formula, i + 1)
            top = stack.pop()
            for a, c in top.items():
                stack[-1][a] += c * k
        else:
            name, i = read_name(formula, i)
            k, i = read_num(formula, i)
            stack[-1][name] += k
    return format_counts(stack[0])

for f, e in EXAMPLES:
    print(f, count_atoms_stack(f), count_atoms_stack(f) == e)

H2O H2O True
Mg(OH)2 H2MgO2 True
K4(ON(SO3)2)2 K4N2O14S4 True


**Code nộp LeetCode (Cách 3).**

```python
from collections import Counter

class Solution:
    def countOfAtoms(self, formula: str) -> str:
        n, i = len(formula), 0
        stack = [Counter()]
        while i < n:
            ch = formula[i]
            if ch == '(':
                stack.append(Counter())
                i += 1
            elif ch == ')':
                i += 1
                j = i
                while i < n and formula[i].isdigit():
                    i += 1
                k = int(formula[j:i]) if i > j else 1
                top = stack.pop()
                for a, c in top.items():
                    stack[-1][a] += c * k
            else:
                j = i
                i += 1
                while i < n and formula[i].islower():
                    i += 1
                name = formula[j:i]
                j = i
                while i < n and formula[i].isdigit():
                    i += 1
                stack[-1][name] += int(formula[j:i]) if i > j else 1
        cnt = stack[0]
        return ''.join(a + (str(cnt[a]) if cnt[a] > 1 else '') for a in sorted(cnt))
```

**Độ phức tạp.** Giống Cách 2: `O(n · k + k log k)` thời gian, `O(n · k)` bộ nhớ trường hợp xấu. Ưu điểm: không lo giới hạn đệ quy.

### 7.3. Cách 4 — Quét ngược với ngăn xếp hệ số

**Giải thích.** Đọc từ **phải sang trái**:

- Gặp chữ số: gom lại thành số `num` (chú ý thứ tự chữ số bị đảo khi đi ngược!).
- Gặp `)`: hệ số của nhóm sắp tới là `num` (mặc định 1). Đẩy `mult × num` vào ngăn xếp hệ số; `mult` = tích hiện tại.
- Gặp `(`: lấy ra khỏi ngăn xếp (thoát nhóm).
- Gặp chữ thường: gom vào tên (tên đọc ngược).
- Gặp chữ hoa: hoàn tất tên; cộng `num × mult` vào bộ đếm duy nhất.

Mỗi nguyên tử được cộng **một lần duy nhất** với đúng tích hệ số — không cần trộn bộ đếm. Thời gian `O(n)` cho phần quét!

In [13]:
def count_atoms_reverse(formula):
    cnt = Counter()
    mult_stack = [1]          # đỉnh = tích hệ số của các ngoặc đang bao quanh
    num, place = 0, 1         # gom số khi đi ngược: chữ số hàng đơn vị đến trước
    name = ""
    for ch in reversed(formula):
        if ch.isdigit():
            num += int(ch) * place
            place *= 10
        elif ch == ")":
            mult_stack.append(mult_stack[-1] * (num or 1))
            num, place = 0, 1
        elif ch == "(":
            mult_stack.pop()
        elif ch.islower():
            name = ch + name
        else:                 # chữ hoa: kết thúc một tên (đọc ngược)
            name = ch + name
            cnt[name] += (num or 1) * mult_stack[-1]
            name, num, place = "", 0, 1
    return format_counts(cnt)

for f, e in EXAMPLES:
    print(f, count_atoms_reverse(f), count_atoms_reverse(f) == e)

H2O H2O True
Mg(OH)2 H2MgO2 True
K4(ON(SO3)2)2 K4N2O14S4 True


> Lưu ý `num or 1`: nếu không có số thì `num == 0` → dùng 1. Điều này an toàn vì đề đảm bảo không có số đếm 0.

**Code nộp LeetCode (Cách 4).**

```python
from collections import Counter

class Solution:
    def countOfAtoms(self, formula: str) -> str:
        cnt = Counter()
        mult = [1]
        num, place, name = 0, 1, ''
        for ch in reversed(formula):
            if ch.isdigit():
                num += int(ch) * place
                place *= 10
            elif ch == ')':
                mult.append(mult[-1] * (num or 1))
                num, place = 0, 1
            elif ch == '(':
                mult.pop()
            elif ch.islower():
                name = ch + name
            else:
                name = ch + name
                cnt[name] += (num or 1) * mult[-1]
                name, num, place = '', 0, 1
        return ''.join(a + (str(cnt[a]) if cnt[a] > 1 else '') for a in sorted(cnt))
```

**Độ phức tạp.** Quét `O(n)` (phép `ch + name` có độ dài tên nhỏ), cộng `O(k log k)` sắp xếp. Bộ nhớ `O(n)` cho ngăn xếp hệ số + `O(k)` bộ đếm.

### 7.4. Cách 5 — Regex tách token + ngăn xếp

**Giải thích.** Dùng `re.findall` để tách công thức thành token một lần, rồi xử lý giống Cách 3. Code ngắn, ít lỗi chỉ số, rất hợp khi phỏng vấn cho phép dùng thư viện chuẩn.

Mỗi kết quả khớp là tuple `(name, count, open, close, close_count)`; chỉ một trong ba nhóm `name`/`open`/`close` khác rỗng.

In [14]:
def count_atoms_regex(formula):
    stack = [Counter()]
    for name, c1, op, cl, c2 in TOKEN.findall(formula):
        if name:
            stack[-1][name] += int(c1 or 1)
        elif op:
            stack.append(Counter())
        else:
            top = stack.pop()
            k = int(c2 or 1)
            for a, c in top.items():
                stack[-1][a] += c * k
    return format_counts(stack[0])

for f, e in EXAMPLES:
    print(f, count_atoms_regex(f), count_atoms_regex(f) == e)

H2O H2O True
Mg(OH)2 H2MgO2 True
K4(ON(SO3)2)2 K4N2O14S4 True


**Code nộp LeetCode (Cách 5).**

```python
import re
from collections import Counter

class Solution:
    def countOfAtoms(self, formula: str) -> str:
        stack = [Counter()]
        for name, c1, op, cl, c2 in re.findall(r'([A-Z][a-z]*)(\d*)|(\()|(\))(\d*)', formula):
            if name:
                stack[-1][name] += int(c1 or 1)
            elif op:
                stack.append(Counter())
            else:
                top = stack.pop()
                k = int(c2 or 1)
                for a, c in top.items():
                    stack[-1][a] += c * k
        cnt = stack[0]
        return ''.join(a + (str(cnt[a]) if cnt[a] > 1 else '') for a in sorted(cnt))
```

**Độ phức tạp.** Như Cách 3: `O(n · k + k log k)`.

## 8. TRACE từng bước

### 8.1. Bảng trạng thái Cách 3 với `K4(ON(SO3)2)2`

| Token | Hành động | Ngăn xếp (từ đáy lên đỉnh) |
|---|---|---|
| `K4` | K += 4 | [{K:4}] |
| `(` | push | [{K:4}, {}] |
| `O` | O += 1 | [{K:4}, {O:1}] |
| `N` | N += 1 | [{K:4}, {O:1,N:1}] |
| `(` | push | [{K:4}, {O:1,N:1}, {}] |
| `S` | S += 1 | [..., {S:1}] |
| `O3` | O += 3 | [..., {S:1,O:3}] |
| `)2` | pop, ×2, cộng xuống | [{K:4}, {O:7,N:1,S:2}] |
| `)2` | pop, ×2, cộng xuống | [{K:4,O:14,N:2,S:4}] |

Kết quả: `K4N2O14S4`.

In [15]:
def trace_stack(formula):
    n, i = len(formula), 0
    stack = [Counter()]
    show = lambda: [dict(c) for c in stack]
    while i < n:
        ch = formula[i]
        if ch == "(":
            stack.append(Counter()); i += 1
            print(f"{'(':<6} push            -> {show()}")
        elif ch == ")":
            start = i
            k, i = read_num(formula, i + 1)
            top = stack.pop()
            for a, c in top.items():
                stack[-1][a] += c * k
            print(f"{formula[start:i]:<6} pop, x{k:<3} cộng   -> {show()}")
        else:
            start = i
            name, i = read_name(formula, i)
            k, i = read_num(formula, i)
            stack[-1][name] += k
            print(f"{formula[start:i]:<6} {name} += {k:<8} -> {show()}")
    return format_counts(stack[0])

print("Kết quả:", trace_stack("K4(ON(SO3)2)2"))

K4     K += 4        -> [{'K': 4}]
(      push            -> [{'K': 4}, {}]
O      O += 1        -> [{'K': 4}, {'O': 1}]
N      N += 1        -> [{'K': 4}, {'O': 1, 'N': 1}]
(      push            -> [{'K': 4}, {'O': 1, 'N': 1}, {}]
S      S += 1        -> [{'K': 4}, {'O': 1, 'N': 1}, {'S': 1}]
O3     O += 3        -> [{'K': 4}, {'O': 1, 'N': 1}, {'S': 1, 'O': 3}]
)2     pop, x2   cộng   -> [{'K': 4}, {'O': 7, 'N': 1, 'S': 2}]
)2     pop, x2   cộng   -> [{'K': 4, 'O': 14, 'N': 2, 'S': 4}]
Kết quả: K4N2O14S4


### 8.2. Trace Cách 4 (quét ngược) với `Mg(OH)2`

Đọc ngược: `2`, `)`, `H`, `O`, `(`, `g`, `M`.

| Ký tự | num | mult_stack | name | Bộ đếm |
|---|---|---|---|---|
| `2` | 2 | [1] | "" | {} |
| `)` | 0 | [1, 2] | "" | {} |
| `H` | 0 | [1, 2] | → H, cộng 1×2 | {H:2} |
| `O` | 0 | [1, 2] | → O, cộng 1×2 | {H:2, O:2} |
| `(` | 0 | [1] | "" | {H:2, O:2} |
| `g` | 0 | [1] | "g" | {H:2, O:2} |
| `M` | 0 | [1] | → Mg, cộng 1×1 | {H:2, O:2, Mg:1} |

In [16]:
def trace_reverse(formula):
    cnt = Counter(); mult = [1]; num, place, name = 0, 1, ""
    print(f"{'ch':<3} {'num':<4} {'mult_stack':<14} {'name':<5} cnt")
    for ch in reversed(formula):
        if ch.isdigit():
            num += int(ch) * place; place *= 10
        elif ch == ")":
            mult.append(mult[-1] * (num or 1)); num, place = 0, 1
        elif ch == "(":
            mult.pop()
        elif ch.islower():
            name = ch + name
        else:
            name = ch + name
            cnt[name] += (num or 1) * mult[-1]
            name, num, place = "", 0, 1
        print(f"{ch:<3} {num:<4} {str(mult):<14} {name:<5} {dict(cnt)}")
    return format_counts(cnt)

print("Kết quả:", trace_reverse("Mg(OH)2"))

ch  num  mult_stack     name  cnt
2   2    [1]                  {}
)   0    [1, 2]               {}
H   0    [1, 2]               {'H': 2}
O   0    [1, 2]               {'H': 2, 'O': 2}
(   0    [1]                  {'H': 2, 'O': 2}
g   0    [1]            g     {'H': 2, 'O': 2}
M   0    [1]                  {'H': 2, 'O': 2, 'Mg': 1}
Kết quả: H2MgO2


In [17]:
# Trace thêm một ví dụ có số nhiều chữ số khi đi ngược
print("Kết quả:", trace_reverse("(H12O)10"))

ch  num  mult_stack     name  cnt
0   0    [1]                  {}
1   10   [1]                  {}
)   0    [1, 10]              {}
O   0    [1, 10]              {'O': 10}
2   2    [1, 10]              {'O': 10}
1   12   [1, 10]              {'O': 10}
H   0    [1, 10]              {'O': 10, 'H': 120}
(   0    [1]                  {'O': 10, 'H': 120}
Kết quả: H120O10


## 9. Code nộp LeetCode

Ta chọn **Cách 3 (ngăn xếp Counter)** làm bản chính: dễ hiểu, không đệ quy, dễ mở rộng. Cell dưới là đúng định dạng LeetCode và được kiểm thử ngay.

In [18]:
from collections import Counter

class Solution:
    def countOfAtoms(self, formula: str) -> str:
        n, i = len(formula), 0
        stack = [Counter()]
        while i < n:
            ch = formula[i]
            if ch == '(':
                stack.append(Counter())
                i += 1
            elif ch == ')':
                i += 1
                j = i
                while i < n and formula[i].isdigit():
                    i += 1
                k = int(formula[j:i]) if i > j else 1
                top = stack.pop()
                for a, c in top.items():
                    stack[-1][a] += c * k
            else:
                j = i
                i += 1
                while i < n and formula[i].islower():
                    i += 1
                name = formula[j:i]
                j = i
                while i < n and formula[i].isdigit():
                    i += 1
                stack[-1][name] += int(formula[j:i]) if i > j else 1
        cnt = stack[0]
        return ''.join(a + (str(cnt[a]) if cnt[a] > 1 else '') for a in sorted(cnt))

sol = Solution()
for f, e in EXAMPLES:
    assert sol.countOfAtoms(f) == e, (f, sol.countOfAtoms(f), e)
print("Solution OK trên ví dụ đề")

Solution OK trên ví dụ đề


## 10. Vì sao thuật toán đúng

**Bổ đề (phân phối).** Nếu nhóm `G = P1 P2 ... Pm` (các phần viết liền nhau) thì số nguyên tử `a` trong `G` bằng tổng số `a` trong từng `Pi`. Nếu `G` có hệ số `k` thì mỗi số được nhân `k`. Đây đúng là định nghĩa của công thức.

**Cách 2/3 đúng** (quy nạp theo độ sâu lồng):

- Phần không có ngoặc: ta cộng trực tiếp `name += k` → đúng.
- Giả sử mọi nhóm có độ sâu < `d` được đếm đúng. Một nhóm độ sâu `d` gồm các phần, mỗi phần là nguyên tố (đúng) hoặc ngoặc có độ sâu < `d` (đúng theo giả thiết) đã được nhân hệ số. Cộng lại theo bổ đề → đúng.

Với ngăn xếp: bất biến "đỉnh ngăn xếp = tổng của phần đã đọc trong nhóm đang mở sâu nhất" được giữ sau mỗi token; khi gặp `)` nhóm đó kết thúc, ta áp hệ số và cộng xuống nhóm cha — đúng y bổ đề.

**Cách 4 đúng.** Theo quan sát 4.3, số đóng góp của mỗi lần xuất hiện nguyên tố = `count × (tích hệ số mọi ngoặc bao quanh)`. Khi quét ngược, lúc gặp một nguyên tố, ngăn xếp hệ số chứa đúng các ngoặc đã gặp `)` mà chưa gặp `(` — tức là các ngoặc **bao quanh** nó; đỉnh là tích của chúng. Do đó mỗi đóng góp được cộng đúng một lần với đúng trọng số.

**Đầu ra** được sắp xếp theo `sorted` → đúng thứ tự từ điển đề yêu cầu; số lượng 1 không in.

## 11. Phân tích Time/Space

Gọi `n` = độ dài chuỗi, `k` = số nguyên tố khác nhau (`k ≤ n/1`).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Khai triển | `O(tổng số nguyên tử)` — có thể ~`10^9` | như time | Chỉ dùng làm trọng tài |
| 2. Đệ quy | `O(n·k + k log k)` | `O(depth·k)` | depth ≤ n/2 |
| 3. Stack Counter | `O(n·k + k log k)` | `O(depth·k)` | Không đệ quy |
| 4. Quét ngược | `O(n + k log k)` | `O(n + k)` | Mỗi nguyên tố cộng 1 lần |
| 5. Regex + stack | `O(n·k + k log k)` | `O(n + depth·k)` | Code ngắn nhất |

Vì sao `O(n·k)` cho Cách 2/3? Mỗi `)` duyệt bộ đếm của nhóm vừa đóng (≤ `k` khoá), có ≤ `n/2` dấu `)`. Thực tế `k` rất nhỏ (số nguyên tố hoá học ~118), nên cả 4 cách tối ưu đều chạy tức thì với `n ≤ 1000`.

In [19]:
import random, time

def worst_nested(depth):
    # ((((...(H2He3Li)2...)2)2
    return "(" * depth + "H2He3Li" + ")2" * depth

f = worst_nested(300)
for fn in [count_atoms_recursive, count_atoms_stack, count_atoms_reverse, count_atoms_regex]:
    t = time.perf_counter(); r = fn(f); t = time.perf_counter() - t
    print(f"{fn.__name__:<24} {t*1000:7.2f} ms  độ dài kết quả {len(r)}")

count_atoms_recursive       0.40 ms  độ dài kết quả 278
count_atoms_stack           0.32 ms  độ dài kết quả 278
count_atoms_reverse         0.07 ms  độ dài kết quả 278
count_atoms_regex           0.37 ms  độ dài kết quả 278


## 12. Bộ test

Ta kiểm tra: (a) ví dụ đề, (b) các trường hợp biên, (c) đối chiếu mọi cách với brute force trên công thức **ngẫu nhiên** có seed (hệ số nhỏ để brute force chạy nổi).

In [20]:
ALL = [count_atoms_brute, count_atoms_recursive, count_atoms_stack,
       count_atoms_reverse, count_atoms_regex, Solution().countOfAtoms]

EDGE = [
    ("H", "H"),
    ("H2", "H2"),
    ("He", "He"),
    ("(H)", "H"),
    ("((H))2", "H2"),
    ("H50", "H50"),
    ("Be32", "Be32"),
    ("H11He49NO35B7N46Li20", "B7H11He49Li20N47O35"),
    ("((N42)24(OB40Li30CHe3O48LiNN26)33(C12Li48N30H13HBe31)21(BHN30Li26BCBe47N40)15(H5)16)14",
     None),  # đáp án: đối chiếu chéo ở cell sau
]
for fn in ALL:
    for f, e in EXAMPLES:
        assert fn(f) == e, (fn.__name__, f)
for fn in ALL[1:]:
    for f, e in EDGE[:-1]:
        assert fn(f) == e, (fn.__name__, f, fn(f), e)
print("Ví dụ đề + edge case OK")

Ví dụ đề + edge case OK


Ca cuối trong `EDGE` là ca lớn từ LeetCode (hệ số tới `14 × 33 × 48`), ta không chép đáp án mà đối chiếu **các cách tối ưu với nhau** (brute force sẽ tạo danh sách hàng chục nghìn phần tử — vẫn chạy được, nên ta đối chiếu luôn).

In [21]:
big = EDGE[-1][0]
answers = {fn.__name__: fn(big) for fn in ALL}
assert len(set(answers.values())) == 1, answers
print(answers["countOfAtoms"])

B18900Be18984C4200H5446He1386Li33894N50106O22638


In [22]:
ELEMENTS = ["H", "He", "Li", "B", "C", "N", "O", "Mg", "Uuo"]

def random_formula(rng, depth=0):
    parts = []
    for _ in range(rng.randint(1, 3)):
        if depth < 3 and rng.random() < 0.35:
            inner = random_formula(rng, depth + 1)
            k = rng.choice([1, 1, 2, 3, 12])
            parts.append("(" + inner + ")" + (str(k) if k > 1 else ""))
        else:
            k = rng.choice([1, 1, 2, 5, 10])
            parts.append(rng.choice(ELEMENTS) + (str(k) if k > 1 else ""))
    return "".join(parts)

rng = random.Random(726)
for _ in range(1500):
    f = random_formula(rng)
    ref = count_atoms_brute(f)
    for fn in ALL[1:]:
        assert fn(f) == ref, (fn.__name__, f, fn(f), ref)
print("1500 công thức ngẫu nhiên: mọi cách khớp brute force. Ví dụ:", f, "->", ref)

1500 công thức ngẫu nhiên: mọi cách khớp brute force. Ví dụ: C10 -> C10


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Chỉ đọc **một** chữ số

**Code sai:** `k = int(formula[i])` → `H12` thành `H1` rồi `2` bị coi như ký tự lạ.

**Vì sao sai:** số đếm có thể nhiều chữ số (`H50`, `)14`).

**Code đúng:** nuốt mọi chữ số liên tiếp bằng vòng `while`.

In [23]:
def read_one_digit_wrong(s, i):
    return (int(s[i]) if i < len(s) and s[i].isdigit() else 1), i + 1

print("sai:", read_one_digit_wrong("H12", 1), " đúng:", read_num("H12", 1))

sai: (1, 2)  đúng: (12, 3)


### Lỗi 2 — Quên hệ số mặc định 1 (hoặc `int("")`)

**Code sai:** `int(formula[j:i])` khi không có chữ số → `ValueError: invalid literal for int() with base 10: ''`.

**Code đúng:** `int(formula[j:i]) if i > j else 1` (hoặc `int(s or 1)`).

In [24]:
try:
    int("")
except ValueError as e:
    print("Lỗi:", e)
print("đúng:", int("" or 1))

Lỗi: invalid literal for int() with base 10: ''
đúng: 1


### Lỗi 3 — Tên nguyên tố chỉ lấy một ký tự

**Code sai:** coi mỗi chữ cái là một nguyên tố → `Mg` thành `M` và `g`.

**Code đúng:** chữ **hoa** mở đầu tên, nuốt tiếp các chữ **thường**.

In [25]:
def names_wrong(s):
    return [ch for ch in s if ch.isalpha()]

def names_right(s):
    return re.findall(r"[A-Z][a-z]*", s)

print(names_wrong("MgUuo"), names_right("MgUuo"))

['M', 'g', 'U', 'u', 'o'] ['Mg', 'Uuo']


### Lỗi 4 — Quét ngược nhưng ghép chữ số sai thứ tự

**Code sai:** `num = num * 10 + int(ch)` khi đi ngược → `12` thành `21`.

**Code đúng:** đi ngược thì chữ số đến trước là hàng **đơn vị**: `num += int(ch) * place; place *= 10`.

In [26]:
def rev_num_wrong(s):
    num = 0
    for ch in reversed(s):
        num = num * 10 + int(ch)
    return num

def rev_num_right(s):
    num, place = 0, 1
    for ch in reversed(s):
        num += int(ch) * place; place *= 10
    return num

print(rev_num_wrong("12"), rev_num_right("12"))

21 12


### Lỗi 5 — Không sắp xếp, hoặc in số `1`

**Code sai:** `"".join(a + str(c) for a, c in cnt.items())` → thứ tự theo lúc chèn và in `Mg1`.

**Code đúng:** `sorted(cnt)` và chỉ in số khi `> 1`.

In [27]:
cnt = Counter({"O": 2, "Mg": 1, "H": 2})
print("sai :", "".join(a + str(c) for a, c in cnt.items()))
print("đúng:", format_counts(cnt))

sai : O2Mg1H2
đúng: H2MgO2


## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu "ngoặc lồng nhau + hệ số/toán tử"** → nghĩ ngay đến **ngăn xếp** hoặc **đệ quy recursive descent**. Các bài cùng họ: *Decode String (394)*, *Basic Calculator (224, 772)*, *Parse Lisp Expression (736)*, *Brace Expansion II (1096)*.
- **Hệ số đứng SAU nhóm** → quét ngược cho phép biết hệ số trước, đơn giản hoá thành một ngăn xếp số.
- **Viết hàm đọc token riêng** (`read_name`, `read_num`) giúp code chính gọn, ít lỗi chỉ số.
- **Ngữ pháp trước, code sau**: viết 3–4 dòng ngữ pháp như mục 4.1 — mỗi luật thành một nhánh `if`.
- **Brute force khai triển** là trọng tài tuyệt vời cho mọi bài parse: rẻ để viết, gần như không thể sai.

## 15. Biến thể

1. **Decode String (LeetCode 394)**: `3[a2[c]]` → `accaccacc`. Cùng cấu trúc: hệ số + ngoặc, nhưng kết quả là chuỗi (không gộp được như Counter).
2. **Công thức có hệ số đứng trước** (`2H2O` kiểu phương trình hoá học): đọc hệ số đầu, nhân cả phân tử.
3. **Ngoặc vuông/ngoặc nhọn** `[ ]`, `{ }`: chỉ cần coi mọi loại mở/đóng như nhau (hoặc kiểm tra khớp).
4. **Tính khối lượng mol**: thay vì in, nhân số lượng với nguyên tử khối và cộng lại.
5. **Kiểm tra cân bằng phương trình** `A + B -> C + D`: đếm hai vế bằng hàm này và so sánh `Counter`.

In [28]:
# Biến thể 4: khối lượng mol (nguyên tử khối làm tròn)
MASS = {"H": 1.008, "C": 12.011, "N": 14.007, "O": 15.999, "S": 32.06, "K": 39.098, "Mg": 24.305}

def molar_mass(formula):
    n, i = len(formula), 0
    stack = [Counter()]
    while i < n:
        if formula[i] == "(":
            stack.append(Counter()); i += 1
        elif formula[i] == ")":
            k, i = read_num(formula, i + 1)
            for a, c in stack.pop().items():
                stack[-1][a] += c * k
        else:
            name, i = read_name(formula, i)
            k, i = read_num(formula, i)
            stack[-1][name] += k
    return sum(MASS[a] * c for a, c in stack[0].items())

print(f"H2O     = {molar_mass('H2O'):.3f}")
print(f"Mg(OH)2 = {molar_mass('Mg(OH)2'):.3f}")

H2O     = 18.015
Mg(OH)2 = 58.319


In [29]:
# Biến thể 5: kiểm tra cân bằng phương trình (hệ số đứng trước phân tử)
def side_counter(side):
    total = Counter()
    for mol in side.split("+"):
        mol = mol.strip()
        m = re.match(r"(\d*)(.*)", mol)
        coef = int(m.group(1) or 1)
        n, i = len(m.group(2)), 0
        f = m.group(2)
        stack = [Counter()]
        while i < n:
            if f[i] == "(":
                stack.append(Counter()); i += 1
            elif f[i] == ")":
                k, i = read_num(f, i + 1)
                for a, c in stack.pop().items():
                    stack[-1][a] += c * k
            else:
                name, i = read_name(f, i); k, i = read_num(f, i)
                stack[-1][name] += k
        for a, c in stack[0].items():
            total[a] += c * coef
    return total

def balanced(eq):
    left, right = eq.split("->")
    return side_counter(left) == side_counter(right)

print(balanced("2H2 + O2 -> 2H2O"), balanced("H2 + O2 -> H2O"))

True False


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Khai triển | Lặp danh sách nguyên tử thật | `O(#nguyên tử)` | `O(#nguyên tử)` | Trọng tài khi test |
| 2. Đệ quy | `parse()` trả Counter, nhân khi đóng ngoặc | `O(n·k)` | `O(depth·k)` | Tự nhiên nhất khi đã có ngữ pháp |
| 3. Stack Counter | Tự quản lý ngăn xếp | `O(n·k)` | `O(depth·k)` | Bản nộp an toàn, không đệ quy |
| 4. Quét ngược | Ngăn xếp hệ số tích luỹ | `O(n)` + sort | `O(n)` | Nhanh nhất, nhưng dễ sai chữ số |
| 5. Regex | Tách token rồi stack | `O(n·k)` | `O(n)` | Code ngắn, ít lỗi chỉ số |

## 17. Bài tập

**Bài 1 (đoán output).** `countOfAtoms("Be32(OH)2(H2O)3")` trả về gì?

**Bài 2 (tính tay).** Lập bảng ngăn xếp (như 8.1) cho `((HO)2N)3`. Kết quả cuối là gì?

**Bài 3 (điền TODO).** Hoàn thiện `read_num_todo(s, i)` trả về `(số, vị_trí_mới)`, mặc định 1 khi không có chữ số.

**Bài 4 (tự viết lại).** Viết lại Cách 4 (quét ngược) trong hàm `my_reverse(formula)` mà không nhìn code.

**Bài 5 (mini challenge).** Viết `decode_string(s)` cho bài *Decode String (394)*: `"3[a2[c]]"` → `"accaccacc"` (hệ số đứng **trước** ngoặc vuông). Dùng ngăn xếp.

In [30]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def read_num_todo(s, i):
    # TODO: j = i; nuốt chữ số; trả (int(s[i:j]) hoặc 1, j)
    pass

# Bài 4 — TODO
def my_reverse(formula):
    # TODO: mult = [1]; duyệt reversed(formula); xử lý digit, ')', '(', thường, hoa
    pass

# Bài 5 — TODO
def decode_string(s):
    # TODO: stack lưu (chuỗi_trước, hệ_số); gặp ']' thì cur = prev + cur * k
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: tách thành `Be32`, `(OH)2`, `(H2O)3`.
- Mức 2: `(OH)2` → O2 H2; `(H2O)3` → H6 O3.
- Mức 3: Be32 H8 O5 → sắp xếp: `Be32H8O5`.

**Bài 2.**
- Mức 1: hai lần push liên tiếp.
- Mức 2: sau `)2` đầu: tầng giữa {H:2, O:2}; thêm N → {H:2, O:2, N:1}.
- Mức 3: `)3` → {H:6, O:6, N:3} → `H6N3O6`.

**Bài 3.**
- Mức 1: dùng `while j < len(s) and s[j].isdigit()`.
- Mức 2: chú ý `j == i` nghĩa là không có số.
- Mức 3: `return (int(s[i:j]) if j > i else 1), j`.

**Bài 4.**
- Mức 1: chữ số đi ngược → hàng đơn vị trước.
- Mức 2: `)` đẩy `mult[-1] * (num or 1)`, reset num.
- Mức 3: chữ hoa: hoàn tất tên, cộng `(num or 1) * mult[-1]`, reset tên và num.

**Bài 5.**
- Mức 1: gặp chữ số → gom `k`; gặp `[` → đẩy `(cur, k)`, reset.
- Mức 2: gặp `]` → `prev, k = stack.pop(); cur = prev + cur * k`.
- Mức 3: gặp chữ cái → `cur += ch`.

## 19. Đáp án tham khảo

In [31]:
# Bài 1, 2
assert Solution().countOfAtoms("Be32(OH)2(H2O)3") == "Be32H8O5"
assert Solution().countOfAtoms("((HO)2N)3") == "H6N3O6"

# Bài 3
def read_num_ans(s, i):
    j = i
    while j < len(s) and s[j].isdigit():
        j += 1
    return (int(s[i:j]) if j > i else 1), j

assert read_num_ans("H12O", 1) == (12, 3)
assert read_num_ans("HO", 1) == (1, 1)
print("Bài 1–3 OK")

Bài 1–3 OK


In [32]:
# Bài 4
def my_reverse_ans(formula):
    cnt, mult = Counter(), [1]
    num, place, name = 0, 1, ""
    for ch in reversed(formula):
        if ch.isdigit():
            num += int(ch) * place; place *= 10
        elif ch == ")":
            mult.append(mult[-1] * (num or 1)); num, place = 0, 1
        elif ch == "(":
            mult.pop()
        elif ch.islower():
            name = ch + name
        else:
            name = ch + name
            cnt[name] += (num or 1) * mult[-1]
            name, num, place = "", 0, 1
    return format_counts(cnt)

rng = random.Random(7)
for _ in range(300):
    f = random_formula(rng)
    assert my_reverse_ans(f) == count_atoms_brute(f)

# Bài 5
def decode_string_ans(s):
    stack, cur, k = [], "", 0
    for ch in s:
        if ch.isdigit():
            k = k * 10 + int(ch)
        elif ch == "[":
            stack.append((cur, k)); cur, k = "", 0
        elif ch == "]":
            prev, m = stack.pop()
            cur = prev + cur * m
        else:
            cur += ch
    return cur

assert decode_string_ans("3[a2[c]]") == "accaccacc"
assert decode_string_ans("2[abc]3[cd]ef") == "abcabccdcdcdef"
assert decode_string_ans("10[a]") == "a" * 10
print("Bài 4–5 OK")

Bài 4–5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template ngăn xếp cho "ngoặc + hệ số đứng sau":**

```python
stack = [Counter()]
while i < n:
    if s[i] == '(':  stack.append(Counter()); i += 1
    elif s[i] == ')':
        k, i = read_num(s, i + 1)
        for a, c in stack.pop().items(): stack[-1][a] += c * k
    else:
        name, i = read_name(s, i); k, i = read_num(s, i)
        stack[-1][name] += k
```

**Template quét ngược:** `mult = [1]`; `)` → push `mult[-1] * (num or 1)`; `(` → pop; chữ hoa → `cnt[name] += (num or 1) * mult[-1]`.

**Checklist trước khi submit:**

- [ ] Đọc số **nhiều chữ số**; không có số → **1**.
- [ ] Tên nguyên tố = 1 chữ hoa + các chữ thường.
- [ ] Gặp `)` thì đọc hệ số **ngay sau** nó.
- [ ] Đầu ra **sắp xếp** theo tên; số lượng 1 **không in**.
- [ ] Nếu đệ quy: kiểm tra độ sâu (≤ 500) hoặc dùng ngăn xếp.
- [ ] Nếu quét ngược: ghép chữ số theo `place`, đảo tên đúng thứ tự.
- [ ] Test: `H`, `((H))2`, `H50`, ví dụ đề, ca lồng sâu.